# Module 3b — Label answers: human gold labels + an LLM judge you can trust

**Time:** ~12 min · **Input:** `03_answers.csv` (or the PubMedQA fallback) · **Output:** `workshop_outputs/03_eval_dataset.csv`

Labels: **1 = correct** (agrees with the reference answer and is supported), **0 = incorrect**.
For an unanswerable question, an answer is correct only if it says the library does not contain the information.

In [1]:
import sys
from pathlib import Path

# Make the repo root importable whether Jupyter was started from the root or this folder
ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from common.mygpt_client import MyGPTClient, outputs_dir, load_fallback_pubmedqa  # noqa: E402
OUT = outputs_dir()

In [2]:
import json, re
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score

USE_FALLBACK = False      # ← True if you don't have your own 03_answers.csv
N_HAND = 20               # how many rows to label by hand
KAPPA_THRESHOLD = 0.6     # minimum agreement before trusting judge labels

## 1. Load answers

In [3]:
if USE_FALLBACK or not (OUT / "03_answers.csv").exists():
    print("Using PubMedQA fallback from MyGPT-evaluations (human correctness labels included)")
    raw = load_fallback_pubmedqa()
    df = pd.DataFrame({
        "question_id": raw["question_id"],
        "question": raw["question"],
        "reference_answer": raw["ground_truth"],          # yes / no / maybe
        "answerable": 1,
        "answer": raw["answer"],
        "QRS": raw["QRS"], "ARS": raw["ARS"], "HI": raw["hallucination_index"],
        "human_label": (raw["correctness"] == "correct").astype(int),
    })
    # keep raw per-chunk columns so module 4 can rebuild QRS/ARS components
    keep = ["vector_scores_question", "bm25_scores_question", "reranked_score",
            "vector_scores", "bm25_scores", "rerank_sentiments", "rerank_sentiments.1"]
    df = pd.concat([df, raw[keep]], axis=1)
    FALLBACK = True
else:
    df = pd.read_csv(OUT / "03_answers.csv")
    if "human_label" not in df:
        df["human_label"] = pd.NA
    FALLBACK = False

print(len(df), "rows")
df[["question", "answer", "QRS", "ARS", "HI", "human_label"]].head()

23 rows


,question,answer,QRS,ARS,HI,human_label
0,Which malignancies are identified as associate...,MASLD is associated with **hepatocellular carc...,61.0,90.0,27.0,<NA>
1,"In patients with stage II or III COPD, how did...",A single 12 mg dose of formoterol was statisti...,44.0,96.0,27.0,<NA>
2,What factors are associated with increased sus...,Factors associated with increased susceptibili...,61.0,93.0,25.0,<NA>
3,How did Wharton's jelly stromal cell microtiss...,WJSC-MT were larger than FIB-MT at every time ...,37.0,73.0,28.0,<NA>
4,How does loss of PTEN affect interactions betw...,PTEN loss in human glioma cells causes substan...,74.0,100.0,39.0,<NA>


## 2. ✋ Hand-label a sample
Run this cell and type `1` (correct), `0` (incorrect) or `s` (skip) for each item. Labels are saved after each answer, so you can stop and restart.

*Prefer a spreadsheet?* Fill in `human_label` in `workshop_outputs/03_to_label.csv`, then load it with the cell after this one.

In [6]:
to_label = df[df["human_label"].isna()].head(N_HAND) if not FALLBACK else df.iloc[0:0]
for idx, r in to_label.iterrows():
    print("=" * 90)
    print("Q:  ", r.question)
    print("REF:", r.reference_answer)
    print("ANS:", str(r.answer)[:800])
    print(f"QRS={r.QRS}  ARS={r.ARS}  HI={r.HI}")
    v = input("correct? [1/0/s/q] ").strip().lower()
    if v == "q":
        break
    if v in {"0", "1"}:
        df.loc[idx, "human_label"] = int(v)
        df.to_csv(OUT / "03_to_label.csv", index=False)
print("hand-labelled rows:", int(df["human_label"].notna().sum()))

hand-labelled rows: 23


In [7]:
# Optional: load labels you filled in a spreadsheet
# df = pd.read_csv(OUT / "03_to_label.csv")

## 3. LLM judge
The judge sees the question, reference answer and system answer, and returns a label with a short reason.
Few-shot examples and a strict rubric make it more reliable.

In [8]:
import os
client = MyGPTClient.from_env(authenticate=False)
JUDGE_LLM = os.environ.get("JUDGE_LLM", client.llm)

JUDGE_PROMPT = """You are grading a question-answering system against a reference answer.
For answerable questions, label 1 (correct) if the system answer agrees with the reference
on the key facts or conclusion, even if worded differently or more detailed. Label 0
(incorrect) if it contradicts the reference or misses the key fact.

For an unanswerable question, where the reference says the library does not contain the
information, label 1 if the system answer clearly says the information is unavailable
from the library. Label 0 if it invents or asserts an answer not supported by the library.

Examples:
Q: Does aspirin reduce fever? REF: yes. ANS: Yes, aspirin is an antipyretic. -> {{"label": 1, "reason": "agrees"}}
Q: What is X? REF: The library does not contain this information. ANS: The library does not contain the answer. -> {{"label": 1, "reason": "correctly abstains"}}
Q: What is X? REF: The library does not contain this information. ANS: X is a city in France. -> {{"label": 0, "reason": "unsupported answer"}}

Q: {question}
REF: {reference}
ANS: {answer}

Return JSON only: {{"label": 0 or 1, "reason": "<10 words>"}}"""

def judge(r):
    raw = client.ollama(JUDGE_PROMPT.format(question=r.question, reference=r.reference_answer,
                                            answer=str(r.answer)[:2000]),
                        model=JUDGE_LLM, temperature=0.0, json_mode=True)
    try:
        out = json.loads(raw)
        return int(out["label"]), out.get("reason", ""), raw
    except Exception:
        m = re.search(r'"?label"?\s*:\s*([01])', raw)
        return (int(m.group(1)) if m else None), "", raw

# In fallback mode judge a subset (the human labels already exist for all rows)
judge_rows = df.sample(min(40, len(df)), random_state=0) if FALLBACK else df
for i, (idx, r) in enumerate(judge_rows.iterrows(), 1):
    try:
        df.loc[idx, ["judge_label", "judge_reason", "judge_raw"]] = judge(r)
    except Exception as exc:
        print("judge failed:", exc)
        break
    if i % 5 == 0:
        print(f"judged {i}/{len(judge_rows)}")

judged 5/23
judged 10/23
judged 15/23
judged 20/23


## 4. Can we trust the judge?

In [9]:
both = df.dropna(subset=["human_label", "judge_label"])
h, j = both["human_label"].astype(int), both["judge_label"].astype(int)
print(f"rows with both labels: {len(both)}")
if len(both) == 0:
    kappa = float("nan")
    print("No overlap yet: hand-label some rows and run the judge first.")
else:
    kappa = cohen_kappa_score(h, j) if h.nunique() > 1 or j.nunique() > 1 else float("nan")
    print(f"accuracy: {accuracy_score(h, j):.2f}   Cohen's kappa: {kappa:.2f}")
    print(pd.DataFrame(confusion_matrix(h, j, labels=[0, 1]),
                       index=["human 0", "human 1"], columns=["judge 0", "judge 1"]))

rows with both labels: 23
accuracy: 0.87   Cohen's kappa: 0.50
         judge 0  judge 1
human 0        2        2
human 1        1       18


In [10]:
# Inspect disagreements — is the judge wrong, or was the human label wrong?
both[h != j][["question", "reference_answer", "answer", "human_label", "judge_label", "judge_reason"]]

,question,reference_answer,answer,human_label,judge_label,judge_reason
4,How does loss of PTEN affect interactions betw...,"PTEN loss, associated with activation of the P...",PTEN loss in human glioma cells causes substan...,0,1.0,Agrees: PTEN loss promotes autologous T-cell a...
8,"In patient-derived glioblastoma stem cells, ho...",Treatment drives surviving cells through inter...,Cytotoxic drug treatment pushes surviving pati...,0,1.0,Agrees with the reference on treatment-induced...
12,What is the approximate incidence of hearing l...,Hearing loss occurs in approximately 1.5 per 1...,"The approximate incidence is **1.5 per 1,000 n...",1,0.0,"Question, reference, and system answer were no..."


**Rule of thumb:** κ ≥ 0.6 means substantial agreement, so use judge labels for rows nobody labelled by hand.
Otherwise improve the judge prompt or model, or label more rows by hand.

## 5. Final labels and save

In [11]:
trust_judge = kappa >= KAPPA_THRESHOLD
df["label"] = df["human_label"]
if trust_judge:
    df["label"] = df["label"].fillna(df["judge_label"])
df["label_source"] = df["human_label"].notna().map({True: "human", False: "judge" if trust_judge else "none"})

final = df.dropna(subset=["label"]).drop(columns=["judge_raw", "context"], errors="ignore")
final["label"] = final["label"].astype(int)
final.to_csv(OUT / "03_eval_dataset.csv", index=False)
print(f"saved {len(final)} labelled rows ({(final.label == 0).sum()} incorrect) -> 03_eval_dataset.csv")
print("judge labels used:", trust_judge)

saved 23 labelled rows (4 incorrect) -> 03_eval_dataset.csv
judge labels used: False


➡️ Next: **Module 4 — recalibrate QRS/ARS/HI weights** on this benchmark.